In [1]:
import os

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

%matplotlib inline
from io import BytesIO

import kagglehub
from PIL import Image
from sklearn import metrics
from tensorflow.keras.layers import (
    Conv2D,
    Dense,
    Dropout,
    Flatten,
    MaxPooling2D,
)
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import plot_model
from tqdm import tqdm

In [ ]:
BATCH_SIZE = 128
IMAGE_SIZE = (224, 224)

In [3]:
path = kagglehub.dataset_download('xhlulu/140k-real-and-fake-faces')
print('Path to dataset files:', path)

base_path = os.path.join(path, 'real_vs_fake', 'real-vs-fake')

Using Colab cache for faster access to the '140k-real-and-fake-faces' dataset.
Path to dataset files: /kaggle/input/140k-real-and-fake-faces


In [4]:
def ela(image_path, scale=(224, 224), quality=90):
    """
    Performs Error Level Analysis (ELA) on an image and returns a 3-channel RGB result.

    Args:
        image_path (str): Path to the image file.
        scale (tuple): Resize dimensions (width, height).
        quality (int): JPEG quality for recompression.

    Returns:
        np.ndarray: 3-channel ELA image in RGB format (uint8).
    """
    # Load and resize image
    image = Image.open(image_path).convert('RGB')
    image = image.resize(scale)

    # Save recompressed image to memory (not disk)
    buffer = BytesIO()
    image.save(buffer, format='JPEG', quality=quality)
    buffer.seek(0)

    compressed = Image.open(buffer)

    # Compute ELA
    diff = np.abs(
        np.array(image, dtype=np.int16) - np.array(compressed, dtype=np.int16)
    )
    diff = np.clip(diff * 10, 0, 255).astype(np.uint8)

    return diff

In [5]:
ela_dir = os.path.join(base_path, 'ela_images')
os.makedirs(ela_dir, exist_ok=True)

for stage in ['train', 'valid', 'test']:
    stage_dir = os.path.join(base_path, stage)
    os.makedirs(stage_dir, exist_ok=True)

    for category in ['real', 'fake']:
        input_dir = os.path.join(stage_dir, category)
        output_dir = os.path.join(ela_dir, stage, category)

        if os.path.exists(output_dir) and os.listdir(output_dir):
            print(f'Skipping {stage}/{category} (already processed)')
            continue

        os.makedirs(output_dir, exist_ok=True)

        for filename in tqdm(
            os.listdir(input_dir), desc=f'Processing {stage}/{category}'
        ):
            if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
                input_path = os.path.join(input_dir, filename)
                output_path = os.path.join(output_dir, filename)
                ela_image = ela(input_path, IMAGE_SIZE)
                Image.fromarray(ela_image).save(output_path)

ela_dir = os.path.join(base_path, 'ela_images')
train_dir = os.path.join(ela_dir, 'train')
val_dir = os.path.join(ela_dir, 'valid')
test_dir = os.path.join(ela_dir, 'test')

OSError: [Errno 30] Read-only file system: '/kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/ela_images'

In [ ]:
datagen = ImageDataGenerator(
    rescale=1.0 / 255,
)

train_generator = datagen.flow_from_directory(
    train_dir,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
)

val_generator = datagen.flow_from_directory(
    val_dir,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
)

test_generator = datagen.flow_from_directory(
    test_dir,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False,
)

Found 100000 images belonging to 2 classes.
Found 20000 images belonging to 2 classes.
Found 20000 images belonging to 2 classes.


In [ ]:
def build_model(input_shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3)):
    model = Sequential(
        [
            Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),
            MaxPooling2D((2, 2)),
            Conv2D(64, (3, 3), activation='relu'),
            MaxPooling2D((2, 2)),
            Conv2D(128, (3, 3), activation='relu'),
            MaxPooling2D((2, 2)),
            Flatten(),
            Dense(128, activation='relu'),
            Dropout(0.5),
            Dense(2, activation='softmax'),
        ]
    )
    optimizer = Adam(learning_rate=0.0001)
    model.compile(
        optimizer=optimizer,
        loss='categorical_crossentropy',
        metrics=[
            'accuracy',
            tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall'),
        ],
    )
    return model


model = build_model()
model.summary()

/home/tiberiu/Documents/Disertatie/140-ela/.venv/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
I0000 00:00:1771708099.719966   13490 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2554 MB memory:  -> device: 0, name: NVIDIA GeForce GTX 1650 Ti, pci bus id: 0000:01:00.0, compute capability: 7.5


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    11,075,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,169,218 (42.61 MB)

 Trainable params: 11,169,218 (42.61 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
plot_model(
    model, to_file='model_architecture.png', show_shapes=True, show_layer_names=True
)

You must install graphviz (see instructions at https://graphviz.gitlab.io/download/) for `plot_model` to work.


In [ ]:
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,
)

Epoch 1/10


2026-02-21 23:08:22.616643: I external/local_xla/xla/service/service.cc:163] XLA service 0x7f30c8004740 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-02-21 23:08:22.616669: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce GTX 1650 Ti, Compute Capability 7.5
2026-02-21 23:08:22.665070: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-02-21 23:08:22.959983: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91900
2026-02-21 23:08:23.770571: I external/local_xla/xla/service/gpu/autotuning/conv_algorithm_picker.cc:546] Omitted potentially buggy algorithm eng14{k25=2} for conv (f32[128,32,222,222]{3,2,1,0}, u8[0]{0}) custom-call(f32[128,3,224,224]{3,2,1,0}, f32[32,3,3,3]{3,2,1,0}, f32[32]{0}), window={size=3x3}, dim_labels=bf01_oi01->bf01, custom_call_target

In [ ]:
if not os.path.exists('models'):
    os.makedirs('models', exist_ok=True)

model.save('models/ela_face_classifier.keras')

# Plotting Accuracy and Loss Graph:

In [ ]:
plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend(['train', 'val'])

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('model Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend(['train', 'val'])
plt.show()

# Confusion Matrix:

In [ ]:
y_pred = model.predict(test_generator)

y_test = test_generator.classes

In [ ]:
y_pred_labels = np.argmax(y_pred, axis=1)

In [ ]:
Fake = False
Real = True
cm_display = metrics.ConfusionMatrixDisplay(
    confusion_matrix=metrics.confusion_matrix(y_test, y_pred_labels),
    display_labels=[Fake, Real],
)

cm_display.plot()
plt.show()

# ROC AUC Score, Precision Score and Test Accuracy:

In [ ]:
print('ROC AUC Score:', metrics.roc_auc_score(y_test, y_pred_labels))
print('AP Score:', metrics.average_precision_score(y_test, y_pred_labels))
print(metrics.classification_report(y_test, y_pred_labels))

In [ ]:
_, accu, _, _ = model.evaluate(test_generator)
print('Final Test Acccuracy = {:.3f}'.format(accu * 100))

# Testing for a random image:

In [ ]:
img_path = os.path.join(ela_dir, 'test', 'real', '00001.jpg')
ela_image = ela(os.path.join(ela_dir, 'test', 'real', '00001.jpg'), IMAGE_SIZE)
p1 = ela_image.astype(np.float32) / 255.0
p1 = np.expand_dims(p1, axis=0)
p1.shape

In [ ]:
op = np.argmax(model.predict(p1), axis=-1)
print(op)
if op == [0]:
    print('Fake Face')
else:
    print('Real Face')